<a href="https://colab.research.google.com/github/despoileralert/DataScience_MachineLearning_Notebooks/blob/main/RandomForest_XGB_DT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import warnings
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import (
    RFE,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    f_classif,
    mutual_info_classif,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_fscore_support

)
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, HalvingGridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from xgboost import XGBClassifier

# Inital Setup

In [2]:
# drive mount
from google.colab import drive
drive.mount('/content/drive')

shared_folder_path = '/content/drive/MyDrive/time-series-human'

# Example: List the files inside the shared folder
import os
print(os.listdir(shared_folder_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['features.txt', 'activity_labels.txt', 'README.txt', 'sampler.py', 'features_info.txt', 'view.ipynb', 'RawData', 'Train', 'Model Training', 'Test', 'feature_extractor.py']


In [3]:
df = pd.read_csv('/content/drive/MyDrive/time-series-human/Train/X_train.txt', sep=r'\s+', header=None)

new_cols = []
with open('/content/drive/MyDrive/time-series-human/features.txt', 'r') as f:
    for line in f:
        new_cols.append(line.strip())
df.columns = new_cols
df

,tBodyAcc-Mean-1,tBodyAcc-Mean-2,tBodyAcc-Mean-3,tBodyAcc-STD-1,tBodyAcc-STD-2,tBodyAcc-STD-3,tBodyAcc-Mad-1,tBodyAcc-Mad-2,tBodyAcc-Mad-3,tBodyAcc-Max-1,...,fBodyGyroJerkMag-MeanFreq-1,fBodyGyroJerkMag-Skewness-1,fBodyGyroJerkMag-Kurtosis-1,tBodyAcc-AngleWRTGravity-1,tBodyAccJerk-AngleWRTGravity-1,tBodyGyro-AngleWRTGravity-1,tBodyGyroJerk-AngleWRTGravity-1,tXAxisAcc-AngleWRTGravity-1,tYAxisAcc-AngleWRTGravity-1,tZAxisAcc-AngleWRTGravity-1
0,0.043580,-0.005970,-0.035054,-0.995381,-0.988366,-0.937382,-0.995007,-0.988816,-0.953325,-0.794796,...,-0.012236,-0.314848,-0.713308,-0.112754,0.030400,-0.464761,-0.018446,-0.841559,0.179913,-0.051718
1,0.039480,-0.002131,-0.029067,-0.998348,-0.982945,-0.971273,-0.998702,-0.983315,-0.974000,-0.802537,...,0.202804,-0.603199,-0.860677,0.053477,-0.007435,-0.732626,0.703511,-0.845092,0.180261,-0.047436
2,0.039978,-0.005153,-0.022651,-0.995482,-0.977314,-0.984760,-0.996415,-0.975835,-0.985973,-0.798477,...,0.440079,-0.404427,-0.761847,-0.118559,0.177899,0.100699,0.808529,-0.849230,0.180610,-0.042271
3,0.039785,-0.011809,-0.028916,-0.996194,-0.988569,-0.993256,-0.996994,-0.988526,-0.993135,-0.798477,...,0.430891,-0.138373,-0.491604,-0.036788,-0.012892,0.640011,-0.485366,-0.848947,0.181907,-0.040826
4,0.038758,-0.002289,-0.023863,-0.998241,-0.986774,-0.993115,-0.998216,-0.986479,-0.993825,-0.801982,...,0.137735,-0.366214,-0.702490,0.123320,0.122542,0.693578,-0.615971,-0.848164,0.185124,-0.037080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7762,0.048048,-0.042445,-0.065884,-0.195448,-0.278326,-0.219954,-0.282233,-0.305861,-0.357803,0.267874,...,-0.008381,-0.596760,-0.879026,-0.190437,0.829718,0.206972,-0.425619,-0.792292,0.238580,0.056020
7763,0.037639,0.006430,-0.044345,-0.235372,-0.302680,-0.232843,-0.322483,-0.354464,-0.345592,0.181271,...,0.209452,-0.404418,-0.684496,0.064907,0.875679,-0.879033,0.400219,-0.772288,0.252653,0.056252
7764,0.037451,-0.002724,0.021009,-0.218281,-0.378082,-0.076950,-0.304446,-0.400661,-0.193071,0.113141,...,0.237003,0.000207,-0.317314,0.052806,-0.266724,0.864404,0.701169,-0.779566,0.249121,0.047071
7765,0.044011,-0.004536,-0.051242,-0.219202,-0.383350,-0.081035,-0.310419,-0.380233,-0.201007,0.166671,...,0.069366,0.037919,-0.356579,-0.101360,0.700740,0.936674,-0.589479,-0.785603,0.246409,0.031700


# Cross Validation Step

In [4]:
#Firstly we handle the train test split

base = "/content/drive/MyDrive/time-series-human"

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

trainingset = pd.read_csv(os.path.join(base, "Train", "features_train.csv")) #Train set
testingset = pd.read_csv(os.path.join(base, "Test", "features_test.csv"))   #Test set

X_train, X_test = trainingset.iloc[:, 3:], testingset.iloc[:, 3:]
y_train, y_test = (trainingset.iloc[:, 2]).to_numpy(), testingset.iloc[:, 2].to_numpy()
Xgroups, ygroups = (trainingset.iloc[:, 0]).to_numpy(), testingset.iloc[:, 0].to_numpy()

print(np.unique(Xgroups))
print(np.unique(ygroups))
print(type(X_train), type(y_train))

#Initialize CV method
sgkf = StratifiedGroupKFold(n_splits=5, shuffle = True, random_state = 42)
sgkf.get_n_splits()

groups = Xgroups
"""print("Xtrain is ", X_train)
print("y_train is ", y_train)"""
for i, (train_index, val_index) in enumerate(sgkf.split(X_train, y_train, Xgroups)):
    print(f"Fold {i}:")
    print(f"  Train: index length = { len(train_index)}")
    print(f"         group={np.unique(groups[train_index])}")
    print(f"  Validation:  index length = {len(val_index)}")
    print(f"         group={np.unique(groups[val_index])}")

[ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
[ 2  4  9 10 12 13 18 20 24]
<class 'pandas.core.frame.DataFrame'> <class 'numpy.ndarray'>
Fold 0:
  Train: index length = 6239
         group=[ 1  3  5  6  8 11 14 15 16 17 21 22 23 26 27 29 30]
  Validation:  index length = 1528
         group=[ 7 19 25 28]
Fold 1:
  Train: index length = 6184
         group=[ 1  3  5  6  7  8 11 14 15 19 21 22 23 25 26 28 29]
  Validation:  index length = 1583
         group=[16 17 27 30]
Fold 2:
  Train: index length = 6024
         group=[ 3  5  6  7 14 16 17 19 21 22 25 26 27 28 29 30]
  Validation:  index length = 1743
         group=[ 1  8 11 15 23]
Fold 3:
  Train: index length = 6363
         group=[ 1  5  6  7  8 11 15 16 17 19 21 23 25 26 27 28 30]
  Validation:  index length = 1404
         group=[ 3 14 22 29]
Fold 4:
  Train: index length = 6258
         group=[ 1  3  7  8 11 14 15 16 17 19 22 23 25 27 28 29 30]
  Validation:  index length = 1509
         group=[ 5  6 21 26]

In [5]:
# We measure the performance of base DT and Random Foreset Models.

X_orig = X_train.loc[:, ~X_train.columns.str.startswith("wavelet")]

models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            n_jobs=-1, random_state=42),
}

featuresets = {"X_orig": X_orig, "X_train": X_train}
results = []

# The Matthews correlation coefficient is used in machine learning as a measure of the quality of binary and multiclass classifications.
# It takes into account true and false positives and negatives and is generally regarded as a balanced measure which can be used even if the classes are of very different sizes.
# The MCC is in essence a correlation coefficient value between -1 and +1.
# A coefficient of +1 represents a perfect prediction, 0 an average random prediction and -1 an inverse prediction.
# The statistic is also known as the phi coefficient. [source: Wikipedia]

scoringmetrics = ["balanced_accuracy", "f1_macro", "matthews_corrcoef"]
#Train on original features + extended features
for feature in featuresets:
  for estname, estimator in models.items():
    s = cross_validate(estimator, featuresets[feature], y_train,
                      groups = Xgroups,
                      scoring = scoringmetrics,
                      cv = sgkf,
                      verbose = 5)
    entry = {"featureset": feature, "model": estname}
    for i in s:
      entry[i] = np.mean(s[i])
    entry["model"] = estname
    results.append(entry)
  print(results)

finaltable = pd.DataFrame(results).round(3)
finaltable

[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.682) matthews_corrcoef: (test=0.769) total time=   7.3s
[CV] END  balanced_accuracy: (test=0.746) f1_macro: (test=0.732) matthews_corrcoef: (test=0.805) total time=   6.7s
[CV] END  balanced_accuracy: (test=0.744) f1_macro: (test=0.741) matthews_corrcoef: (test=0.853) total time=   6.7s
[CV] END  balanced_accuracy: (test=0.648) f1_macro: (test=0.652) matthews_corrcoef: (test=0.784) total time=   9.2s
[CV] END  balanced_accuracy: (test=0.620) f1_macro: (test=0.604) matthews_corrcoef: (test=0.793) total time=   6.4s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   36.3s finished


[CV] END  balanced_accuracy: (test=0.764) f1_macro: (test=0.743) matthews_corrcoef: (test=0.853) total time=  44.5s
[CV] END  balanced_accuracy: (test=0.827) f1_macro: (test=0.833) matthews_corrcoef: (test=0.909) total time=  43.3s
[CV] END  balanced_accuracy: (test=0.822) f1_macro: (test=0.830) matthews_corrcoef: (test=0.950) total time=  40.9s
[CV] END  balanced_accuracy: (test=0.766) f1_macro: (test=0.763) matthews_corrcoef: (test=0.892) total time=  45.1s
[CV] END  balanced_accuracy: (test=0.745) f1_macro: (test=0.762) matthews_corrcoef: (test=0.848) total time=  44.2s
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(7.242588329315185), 'score_time': np.float64(0.015192556381225585), 'test_balanced_accuracy': np.float64(0.6891547262951427), 'test_f1_macro': np.float64(0.6822009341250407), 'test_matthews_corrcoef': np.float64(0.8006883543170892)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(43.46476120948792), 'score_time': np.

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  3.6min finished


[CV] END  balanced_accuracy: (test=0.747) f1_macro: (test=0.723) matthews_corrcoef: (test=0.790) total time=  17.2s
[CV] END  balanced_accuracy: (test=0.760) f1_macro: (test=0.747) matthews_corrcoef: (test=0.812) total time=  13.7s
[CV] END  balanced_accuracy: (test=0.820) f1_macro: (test=0.821) matthews_corrcoef: (test=0.887) total time=  15.7s
[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.686) matthews_corrcoef: (test=0.756) total time=  18.9s
[CV] END  balanced_accuracy: (test=0.754) f1_macro: (test=0.739) matthews_corrcoef: (test=0.798) total time=  17.2s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  1.4min finished


[CV] END  balanced_accuracy: (test=0.811) f1_macro: (test=0.803) matthews_corrcoef: (test=0.855) total time=  58.3s
[CV] END  balanced_accuracy: (test=0.836) f1_macro: (test=0.847) matthews_corrcoef: (test=0.911) total time= 1.2min
[CV] END  balanced_accuracy: (test=0.865) f1_macro: (test=0.865) matthews_corrcoef: (test=0.958) total time=  59.8s
[CV] END  balanced_accuracy: (test=0.817) f1_macro: (test=0.814) matthews_corrcoef: (test=0.894) total time= 1.0min
[CV] END  balanced_accuracy: (test=0.838) f1_macro: (test=0.820) matthews_corrcoef: (test=0.849) total time= 1.0min
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(7.242588329315185), 'score_time': np.float64(0.015192556381225585), 'test_balanced_accuracy': np.float64(0.6891547262951427), 'test_f1_macro': np.float64(0.6822009341250407), 'test_matthews_corrcoef': np.float64(0.8006883543170892)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(43.46476120948792), 'score_time': np.

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  5.2min finished


,featureset,model,fit_time,score_time,test_balanced_accuracy,test_f1_macro,test_matthews_corrcoef
0,X_orig,Decision tree,7.243,0.015,0.689,0.682,0.801
1,X_orig,Random forest,43.465,0.131,0.785,0.786,0.890
2,X_train,Decision tree,16.511,0.022,0.754,0.743,0.809
3,X_train,Random forest,62.444,0.150,0.833,0.830,0.894


# Correlation Filter + Dimensionality Reduction + Model Training

In [6]:
!pip install feature_engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 5.9 MB/s eta 0:00:00


Various transform steps to be used in the pipeline later on

In [7]:
from sklearn.feature_selection import SelectFromModel
from feature_engine.selection import DropCorrelatedFeatures, SmartCorrelatedSelection
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

#Other DimReduction transformers to compare against
corrFilter = SmartCorrelatedSelection(method="pearson", selection_method="variance")

PCAtransform = PCA(random_state=42)
FAtransform = FactorAnalysis(random_state=42)
ICAtransform = FastICA(whiten="unit-variance", max_iter=500, random_state=42)
LDAtransform = LinearDiscriminantAnalysis()
GRPtransform = GaussianRandomProjection(random_state=42)
selector = SelectFromModel(estimator=ExtraTreesClassifier(random_state=42, n_estimators = 100))

In [14]:
n_components_options = [100, 150, 200]

corr_params = {
    "corr__threshold": [0.9, 0.95],
}

transform_param_grid = [
    {
        "reduce_dim": [PCAtransform, FAtransform, GRPtransform, ICAtransform],
        "reduce_dim__n_components": n_components_options,
    },
    {
        "reduce_dim": [LDAtransform],
        "reduce_dim__n_components": [11],
    },
    {
        "reduce_dim": [selector],
        "reduce_dim__max_features": [100, 200, 300],
    }
]
reducer_labels = ["PCA", "FA", "Model Selection", "ICA", "LDA", "GRP"]

fulltransformgrid = [{**corr_params, **g} for g in transform_param_grid]

RFpipe = Pipeline(
    [
        ("corr", corrFilter),
        ("reduce_dim", "passthrough"),
        ("classify", RandomForestClassifier(random_state = 42)),
    ],
    memory = "pipeline_cache"
)

DTpipe = Pipeline(
    [
        ("corr", corrFilter),
        ("reduce_dim", "passthrough"),
        ("classify", DecisionTreeClassifier(random_state = 42)),
    ],
    memory = "pipeline_cache"
)

XGBpipe = Pipeline(
    [
        ("corr", corrFilter),
        ("reduce_dim", "passthrough"),
        ("classify", XGBClassifier(random_state = 42, tree_method = "hist")),
    ],
    memory = "pipeline_cache"
)

y_train_xgb = y_train - 1

In [ ]:
RFmodel = HalvingGridSearchCV(RFpipe, param_grid=fulltransformgrid, cv=sgkf, scoring="f1_macro",
                              refit=True, verbose=10, random_state = 42)
DTmodel = HalvingGridSearchCV(DTpipe, param_grid=fulltransformgrid, cv=sgkf, scoring="f1_macro",
                              refit=True, verbose=10, random_state = 42)
XGBmodel = GridSearchCV(XGBpipe, param_grid=fulltransformgrid, cv=sgkf, scoring="f1_macro",
                              refit=True, verbose=10)

RFmodel.fit(X_train, y_train, groups=Xgroups)
DTmodel.fit(X_train, y_train, groups=Xgroups)
XGBmodel.fit(X_train, y_train_xgb, groups=Xgroups)

In [19]:
print(RFmodel.best_params_, RFmodel.best_score_, RFmodel.best_estimator_)
print()
print(DTmodel.best_params_, DTmodel.best_score_, DTmodel.best_estimator_)
print()
print(XGBmodel.best_params_, XGBmodel.best_score_, XGBmodel.best_estimator_)

{'corr__threshold': 0.95, 'reduce_dim': SelectFromModel(estimator=ExtraTreesClassifier(random_state=42)), 'reduce_dim__max_features': 200} 0.8167944229541273 Pipeline(memory='pipeline_cache',
         steps=[('corr',
                 SmartCorrelatedSelection(selection_method='variance',
                                          threshold=0.95)),
                ('reduce_dim',
                 SelectFromModel(estimator=ExtraTreesClassifier(random_state=42),
                                 max_features=200)),
                ('classify', RandomForestClassifier(random_state=42))])

{'corr__threshold': 0.95, 'reduce_dim': SelectFromModel(estimator=ExtraTreesClassifier(random_state=42)), 'reduce_dim__max_features': 200} 0.7362253630995232 Pipeline(memory='pipeline_cache',
         steps=[('corr',
                 SmartCorrelatedSelection(selection_method='variance',
                                          threshold=0.95)),
                ('reduce_dim',
                 SelectFromModel(e

In [ ]:
RFmodel2 = GridSearchCV(
    clone(RFmodel.best_estimator_),
    param_grid={
        "classify__max_features": ["sqrt", "log2", 0.3, 0.5],
        "classify__min_samples_leaf": [3, 5, 7, 9],
    },
    cv=sgkf, scoring="f1_macro", verbose=5
)

DTmodel2 = GridSearchCV(
    clone(DTmodel.best_estimator_),
    param_grid={
        "classify__max_depth": [10, 20, None],
        "classify__min_samples_leaf": [3, 5, 7, 9],
        "classify__ccp_alpha": [0.0, 1e-3]
    },
    cv=sgkf, scoring="f1_macro", verbose=5)

XGBmodel2 = GridSearchCV(
    clone(XGBmodel.best_estimator_),
    param_grid={
        "classify__learning_rate": [0.05, 0.1],
        "classify__n_estimators": [300, 600],
        "classify__max_depth": [4,6],
        "classify__colsample_by_tree": [0.4, 0.8]
        }
    ,cv=sgkf, scoring="f1_macro", verbose=5)


RFmodel2.fit(X_train, y_train, groups=Xgroups)
DTmodel2.fit(X_train, y_train, groups=Xgroups)
XGBmodel2.fit(X_train, y_train_xgb, groups=Xgroups)

In [21]:
print(RFmodel2.best_params_, RFmodel2.best_score_, RFmodel2.best_estimator_)
print()
print(DTmodel2.best_params_, DTmodel2.best_score_, DTmodel2.best_estimator_)
print()
print(XGBmodel2.best_params_, XGBmodel2.best_score_, XGBmodel2.best_estimator_)

{'classify__max_features': 0.3, 'classify__min_samples_leaf': 5} 0.8319487639639067 Pipeline(memory='pipeline_cache',
         steps=[('corr',
                 SmartCorrelatedSelection(selection_method='variance',
                                          threshold=0.95)),
                ('reduce_dim',
                 SelectFromModel(estimator=ExtraTreesClassifier(random_state=42),
                                 max_features=200)),
                ('classify',
                 RandomForestClassifier(max_features=0.3, min_samples_leaf=5,
                                        random_state=42))])

{'classify__ccp_alpha': 0.001, 'classify__max_depth': 20, 'classify__min_samples_leaf': 5} 0.7460777292516959 Pipeline(memory='pipeline_cache',
         steps=[('corr',
                 SmartCorrelatedSelection(selection_method='variance',
                                          threshold=0.95)),
                ('reduce_dim',
                 SelectFromModel(estimator=ExtraTreesClassifi

In [22]:
final_RF = RFmodel2.best_estimator_
y_pred = final_RF.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print("Test precision:", precision_score(y_test, y_pred, average = 'macro'))
print()
print("Test recall:", recall_score(y_test, y_pred, average = 'macro'))
print()
print("Test f1:", f1_score(y_test, y_pred, average = 'macro'))
print()

print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.9278937381404174

Test precision: 0.8554203360500329

Test recall: 0.8582773154661661

Test f1: 0.8540928206878281

              precision    recall  f1-score   support

           1       0.95      0.98      0.97       496
           2       0.94      0.95      0.95       471
           3       0.98      0.93      0.95       420
           4       0.91      0.83      0.87       508
           5       0.86      0.93      0.90       556
           6       1.00      1.00      1.00       545
           7       0.72      0.78      0.75        23
           8       0.83      1.00      0.91        10
           9       0.67      0.81      0.73        32
          10       0.86      0.76      0.81        25
          11       0.75      0.61      0.67        49
          12       0.79      0.70      0.75        27

    accuracy                           0.93      3162
   macro avg       0.86      0.86      0.85      3162
weighted avg       0.93      0.93      0.93      3162



In [25]:
final_DT = DTmodel2.best_estimator_
y_pred = final_DT.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print("Test precision:", precision_score(y_test, y_pred, average = 'macro'))
print()
print("Test recall:", recall_score(y_test, y_pred, average = 'macro'))
print()
print("Test f1:", f1_score(y_test, y_pred, average = 'macro'))
print()

print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.8627450980392157

Test precision: 0.735371659658695

Test recall: 0.7104229272749979

Test f1: 0.7124018982604277

              precision    recall  f1-score   support

           1       0.80      0.92      0.85       496
           2       0.91      0.83      0.87       471
           3       0.90      0.84      0.87       420
           4       0.87      0.79      0.83       508
           5       0.82      0.89      0.85       556
           6       1.00      0.99      1.00       545
           7       0.83      0.43      0.57        23
           8       0.50      0.70      0.58        10
           9       0.45      0.56      0.50        32
          10       0.58      0.44      0.50        25
          11       0.63      0.45      0.52        49
          12       0.55      0.67      0.60        27

    accuracy                           0.86      3162
   macro avg       0.74      0.71      0.71      3162
weighted avg       0.87      0.86      0.86      3162

C

In [26]:
final_XGB = XGBmodel2.best_estimator_
y_pred = final_XGB.predict(X_test)
print("Test accuracy:", accuracy_score(y_test-1, y_pred))
print()
print("Test precision:", precision_score(y_test-1, y_pred, average = 'macro'))
print()
print("Test recall:", recall_score(y_test-1, y_pred, average = 'macro'))
print()
print("Test f1:", f1_score(y_test-1, y_pred, average = 'macro'))
print()

print(classification_report(y_test-1, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test-1, y_pred))

Test accuracy: 0.9528779253636939

Test precision: 0.8792348157792117

Test recall: 0.8803735252385244

Test f1: 0.8759570404554382

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       496
           1       0.98      0.99      0.99       471
           2       1.00      0.99      0.99       420
           3       0.92      0.88      0.90       508
           4       0.90      0.94      0.92       556
           5       1.00      1.00      1.00       545
           6       0.79      0.96      0.86        23
           7       0.90      0.90      0.90        10
           8       0.72      0.91      0.81        32
           9       0.74      0.68      0.71        25
          10       0.91      0.63      0.75        49
          11       0.70      0.70      0.70        27

    accuracy                           0.95      3162
   macro avg       0.88      0.88      0.88      3162
weighted avg       0.95      0.95      0.95      3162

